# 04 · Labelling (zero-shot)

Give each label a short description. `eg.classify` gives every item the label whose
description is closest. No training is needed.

In [1]:
import os
from pathlib import Path

import embeddinggemma_engine as eg

if Path.cwd().name == "notebooks":
    os.chdir("..")  # run from the repo root, so paths stay short
IMAGES = Path("assets/images")
VIDEOS = Path("assets/videos")
AUDIO = Path("assets/audio")


def mistake(fn):
    """Run fn and print the error the engine raises for it."""
    try:
        fn()
    except eg.EngineError as error:
        print(f"{type(error).__name__}: {error}\n")

In [2]:
engine = eg.EmbeddingEngine()

## Label some photos

In [3]:
labels = {
    "food": "a photo of a meal",
    "shop": "a photo of products on shop shelves",
    "animals": "a photo of wild animals",
    "people": "a photo of a person",
    "waste": "a photo of rubbish or pollution",
}
photos = ["dishes.jpg", "merchandise2.jpg", "wildlife.jpg", "human.jpg", "trash.jpg", "girl.jpg"]

results = eg.classify(engine, [eg.Item.image(IMAGES / p) for p in photos], labels=labels)
for photo, result in zip(photos, results):
    print(f"{photo:18} {result.name:8} score {result.score:.2f}")

dishes.jpg         food     score 0.67
merchandise2.jpg   shop     score 0.75
wildlife.jpg       animals  score 0.73
human.jpg          people   score 0.66
trash.jpg          waste    score 0.74
girl.jpg           people   score 0.60


## How sure is it?
Each result also has the score for **every** label. The gap between the best and the
second-best label is a good confidence signal. Send small gaps to a human.

In [4]:
for photo, result in zip(photos, results):
    second = sorted(result.scores.values())[-2]
    gap = result.score - second
    print(f"{photo:18} {result.name:8} gap {gap:.3f}  {'check by hand' if gap < 0.02 else ''}")

dishes.jpg         food     gap 0.059  
merchandise2.jpg   shop     gap 0.090  
wildlife.jpg       animals  gap 0.115  
human.jpg          people   gap 0.064  
trash.jpg          waste    gap 0.079  
girl.jpg           people   gap 0.035  


## Videos and audio
It works the same for any media, and the descriptions just need to fit it.

In [5]:
clips = ["01_dog.mp4", "03_blocks.mp4", "04_coffee.mp4", "05_juggle.mp4"]
activities = {
    "pet": "a person playing with a pet",
    "game": "a table-top game or puzzle",
    "drink": "preparing a drink",
    "sport": "a person doing ball tricks",
}
for clip, result in zip(clips, eg.classify(engine, [eg.Item.video(VIDEOS / c) for c in clips], labels=activities)):
    print(f"{clip:14} {result.name}")

01_dog.mp4     pet
03_blocks.mp4  game
04_coffee.mp4  drink
05_juggle.mp4  sport


In [6]:
sounds = {
    "soda can": "voicebosch-soda-can-opening-167415.mp3",
    "flamenco": "ahmadmousavipour-1throw-the-chord-flamenco-stock-299645.mp3",
    "halloween": "dragon-studio-halloween-creepy-stock-sound-effect-511328.mp3",
    "guitar loop": "van_wiese-gitarren-loop-136bpm-stereo-8969.mp3",
}
kinds = {"music": "a piece of music", "sound effect": "a short sound effect"}
for name, result in zip(sounds, eg.classify(engine, [eg.Item.audio(AUDIO / f) for f in sounds.values()], labels=kinds)):
    print(f"{name:12} {result.name}")

soda can     sound effect
flamenco     music
halloween    sound effect
guitar loop  music


## Tips
- Write descriptions the way you would caption the item ("a photo of…", "a recording of…").
- `classify` uses the `similarity` task by default. It did best in testing (see `research/`).
  You can pass `task="classification"` or `task="clustering"` to compare.
- For many items, embed the label descriptions once and reuse them: `classify` is a thin
  wrapper around `engine.embed` + `eg.similarity`.

## Common mistakes

In [7]:
print("Only one label (there is nothing to choose between):")
mistake(lambda: eg.classify(engine, ["x"], labels={"food": "a meal"}))

print("A query-style task:")
mistake(lambda: eg.classify(engine, ["x"], labels=labels, task="search"))

Only one label (there is nothing to choose between):
InputError: classify needs at least two labels.

A query-style task:
TaskError: search is asymmetric: use embed_queries / embed_documents, not embed.

